In [1]:
import pandas as pd
df = pd.read_csv("Dataset.csv")
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
print("Dataset Shape:", df.shape)
print("\nColumn Information:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

Dataset Shape: (8790, 10)

Column Information:
<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 1.6 MB

Missing Values:
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

Duplicate Rows: 0


In [3]:
features = ['type', 'director', 'country', 'rating', 'listed_in']
df['content'] = df[features].fillna('').agg(' '.join, axis=1)
df[['title', 'content']].head()

,title,content
0,Dick Johnson Is Dead,Movie Kirsten Johnson United States PG-13 Docu...
1,Ganglands,TV Show Julien Leclercq France TV-MA Crime TV ...
2,Midnight Mass,TV Show Mike Flanagan United States TV-MA TV D...
3,Confessions of an Invisible Girl,Movie Bruno Garotti Brazil TV-PG Children & Fa...
4,Sankofa,"Movie Haile Gerima United States TV-MA Dramas,..."


In [5]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 210.4 kB/s eta 0:00:38
   -- --------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (8790, 6569)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(tfidf_matrix)
print("Similarity Matrix Shape:", similarity_matrix.shape)

Similarity Matrix Shape: (8790, 8790)


In [8]:
def recommend(title, num_recommendations=10):
    # Find the index of the selected title
    index = df[df['title'].str.lower() == title.lower()].index[0]
    # Get similarity scores for this title
    similarity_scores = list(enumerate(similarity_matrix[index]))
    # Sort by similarity score
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    # Get the top recommendations
    recommendations = similarity_scores[1:num_recommendations + 1]
    # Display recommended titles
    print(f"Recommendations for: {df.loc[index, 'title']}\n")
    for i, score in recommendations:
        print(f"{df.loc[i, 'title']}  | Similarity: {score:.2f}")

In [9]:
recommend("Stranger Things")

Recommendations for: Stranger Things

Stranger Things  | Similarity: 1.00
Nightflyers  | Similarity: 0.97
Helix  | Similarity: 0.97
Manifest  | Similarity: 0.93
The Vampire Diaries  | Similarity: 0.93
The 4400  | Similarity: 0.93
The Messengers  | Similarity: 0.93
The OA  | Similarity: 0.90
The Umbrella Academy  | Similarity: 0.89
Warrior Nun  | Similarity: 0.85


In [10]:
recommend("The Crown")

Recommendations for: The Crown

The Crown  | Similarity: 1.00
Black Mirror  | Similarity: 1.00
Traitors  | Similarity: 1.00
Black Earth Rising  | Similarity: 1.00
Requiem  | Similarity: 1.00
Sex Education  | Similarity: 0.96
Cuckoo  | Similarity: 0.96
Derry Girls  | Similarity: 0.96
The Duchess  | Similarity: 0.96
The Windsors  | Similarity: 0.96


In [11]:
recommend("Breaking Bad")

Recommendations for: Breaking Bad

Dare Me  | Similarity: 1.00
Ozark  | Similarity: 1.00
Designated Survivor  | Similarity: 1.00
Breaking Bad  | Similarity: 1.00
The Blacklist  | Similarity: 0.95
The Lizzie Borden Chronicles  | Similarity: 0.95
Hap and Leonard  | Similarity: 0.92
StartUp  | Similarity: 0.92
Unbelievable  | Similarity: 0.92
When They See Us  | Similarity: 0.92


In [12]:
def evaluate_recommendations(title):
    index = df[df['title'].str.lower() == title.lower()].index[0]
    scores = list(enumerate(similarity_matrix[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    recommendations = scores[1:11]
    original_genres = set(df.loc[index, 'listed_in'].split(', '))
    total = 0

    for i, score in recommendations:
        genres = set(df.loc[i, 'listed_in'].split(', '))
        common = original_genres.intersection(genres)
        if len(common) > 0:
            total += 1
    accuracy = total / len(recommendations) * 100

    print("Title:", df.loc[index, 'title'])
    print("Recommendations with common genres:", total)
    print("Simple evaluation score:", round(accuracy, 2), "%")

In [15]:
evaluate_recommendations("Breaking Bad")

Title: Breaking Bad
Recommendations with common genres: 10
Simple evaluation score: 100.0 %


In [16]:
evaluate_recommendations("Stranger Things")

Title: Stranger Things
Recommendations with common genres: 10
Simple evaluation score: 100.0 %


In [17]:
evaluate_recommendations("The Crown")

Title: The Crown
Recommendations with common genres: 10
Simple evaluation score: 100.0 %


## Conclusion

In this project, I built a content-based recommendation system for Netflix titles.

I used content features such as type, director, country, rating, and genres. 
I converted the text features into numerical values using TF-IDF and used Cosine Similarity to find similar titles.

The system can recommend the top 10 similar titles for a selected movie or TV show.

I also tested the recommendation system on different titles and evaluated the recommendations based on shared genres.